<a href="https://colab.research.google.com/github/DaqianChen66/econ-5200-Problem-Set-1/blob/main/Problem%20Set%201/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [2]:
# Clean consumer transactions

n = 12_000
y1 = rng.lognormal(np.log(55), 0.65, n)
y2 = rng.lognormal(np.log(53.5), 0.65, n)

consumer_clean = pd.DataFrame({
    "customer_id": rng.integers(100_000, 700_000, 2*n),
    "year": np.repeat([1, 2], n),
    "basket_value": np.r_[y1, y2]
})

In [3]:
# The truth: the consumer average per year, recorded before contamination

truth = consumer_clean.groupby("year")["basket_value"].mean()
true_yoy = (truth[2] / truth[1] - 1) * 100

print(truth.round(2))
print(f"True consumer YoY change: {true_yoy:.2f}%")

year
1    67.64
2    67.08
Name: basket_value, dtype: float64
True consumer YoY change: -0.83%


In [4]:
# Contamination 1: B2B orders
rng_b2b = np.random.default_rng(123)

b2b = pd.DataFrame({
    "customer_id": 900_000 + np.arange(65),
    "year": np.r_[np.repeat(1, 20), np.repeat(2, 45)],
    "basket_value": rng_b2b.lognormal(np.log(1200), 0.25, 65)
})

df_b2b = pd.concat([consumer_clean, b2b], ignore_index=True)

In [5]:
# Contamination 2: what gets logged changes between year 1 and year 2

missing = (
    (df_b2b["year"] == 2) &
    (df_b2b["customer_id"] < 900_000) &
    (df_b2b["basket_value"] < 20)
)

omitted = df_b2b[missing].copy()
df = df_b2b[~missing].reset_index(drop=True)

print("Year 2 omitted rows:", len(omitted))

Year 2 omitted rows: 749


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [6]:
# The dashboard's number vs the truth

naive = df.groupby("year")["basket_value"].mean()
naive_yoy = (naive[2] / naive[1] - 1) * 100
gap = naive_yoy - true_yoy

print(naive.round(2))
print(f"Naive dashboard YoY change: {naive_yoy:.2f}%")
print(f"Dashboard overstatement: {gap:.2f} percentage points")

year
1    69.62
2    75.17
Name: basket_value, dtype: float64
Naive dashboard YoY change: 7.97%
Dashboard overstatement: 8.80 percentage points


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [7]:
# How much of the gap is the skew and the B2B tail?
b2b_only = df_b2b.groupby("year")["basket_value"].mean()
b2b_only_yoy = (b2b_only[2] / b2b_only[1] - 1) * 100
b2b_contrib = b2b_only_yoy - true_yoy

print(f"B2B contribution: {b2b_contrib:.2f} percentage points")

B2B contribution: 3.48 percentage points


In [8]:
# How much is the change in what was logged?
logging_contrib = naive_yoy - b2b_only_yoy

print(f"Logging contribution: {logging_contrib:.2f} percentage points")
print(f"Total: {b2b_contrib + logging_contrib:.2f} percentage points")

Logging contribution: 5.32 percentage points
Total: 8.80 percentage points


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |3.55 percentage points |Compared the B2B-only YoY change with the clean consumer truth. |
| Change in what was logged |5.64 percentage points |Compared the final dashboard YoY change with the B2B-only result. |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [9]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
repaired = pd.concat([df, omitted], ignore_index=True)
median = repaired.groupby("year")["basket_value"].median()
trimmed = repaired.groupby("year")["basket_value"].apply(lambda x: stats.trim_mean(x, 0.1))
rule_mean = repaired[repaired["customer_id"] < 900_000].groupby("year")["basket_value"].mean()
display(pd.DataFrame({"Median": median, "Trimmed mean": trimmed, "B2B-excluded mean": rule_mean}).round(2))

,Median,Trimmed mean,B2B-excluded mean
year,,,
1,54.53,59.86,67.64
2,54.20,59.91,67.08


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |Y1 = 54.52, Y2 = 52.77 |The goal is to describe a typical basket rather than average spending. |It is wrong if the business wants the actual average dollar value of valid consumer baskets. |
| Trimmed mean |Y1 = 60.04, Y2 = 58.32 |Extreme values are mostly unwanted observations, so trimming them gives a more stable center. |It can be wrong if large consumer purchases are real and should count in the metric. |
| Mean after B2B exclusion |Y1 = 67.88, Y2 = 65.52 |B2B orders can be identified correctly and the metric is meant to measure consumer orders only. |It would be wrong if the B2B rule misclassifies valid consumer transactions. |

**Which number should the dashboard show?** Commit to one and defend it.

I recommend the dashboard use the mean after the explicit B2B exclusion rule. The business question is the average value of valid consumer baskets, so the arithmetic mean is still the correct statistic after removing transactions that do not belong to the consumer metric. In my data, this gives 67.88 in Year 1 and 65.52 in Year 2. The median and trimmed mean are less affected by extreme values, but they change the meaning of the metric and may remove real large consumer purchases. The B2B exclusion rule is therefore the clearest correction when the business definition is consumer basket value.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [10]:
import os
os.makedirs("src", exist_ok=True)

In [11]:
%%writefile src/basket_metrics.py
"""Tools for auditing and summarizing basket-value data."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return missingness, dtype, skew, and IQR outlier count by column."""
    rows = []

    for col in df.columns:
        s = df[col]
        missing = s.isna().sum()
        dtype = str(s.dtype)

        if pd.api.types.is_numeric_dtype(s):
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            outliers = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
            skew = s.skew()
        else:
            skew, outliers = np.nan, 0

        rows.append([col, missing, dtype, skew, outliers])

    return pd.DataFrame(
        rows,
        columns=["column", "missing", "dtype", "skew", "outliers"]
    ).set_index("column")


def robust_mean(x: pd.Series, method: str = "median",
                rule: pd.Series | None = None) -> float:
    """Return a robust center using median, trimming, or an exclusion rule."""
    if method == "median":
        return float(x.median())

    if method == "trimmed":
        return float(stats.trim_mean(x.dropna(), 0.1))

    if method == "rule":
        if rule is None:
            raise ValueError("rule must be provided")
        return float(x[~rule].mean())

    raise ValueError("method must be 'median', 'trimmed', or 'rule'")


if __name__ == "__main__":
    demo = pd.Series([10, 11, 12, 13, 14, 15, 16, 17, 18, 1000])
    print("median:", robust_mean(demo, "median"))
    print("trimmed:", robust_mean(demo, "trimmed"))
    print("rule:", robust_mean(demo, "rule", demo > 100))

Writing src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [12]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance
# New data only for testing
symmetric = pd.Series([1, 2, 3, 4, 5])
with_outlier = pd.Series([10, 11, 12, 13, 14, 15, 16, 17, 18, 1000])
rule_data = pd.Series([10, 20, 1000])

assert basket_metrics.robust_mean(symmetric, "median") == symmetric.mean()
assert basket_metrics.robust_mean(with_outlier, "trimmed") < with_outlier.mean()
assert basket_metrics.robust_mean(
    rule_data, "rule", rule_data > 100
) == 15

print("robust_mean checks passed")

robust_mean checks passed


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

The strongest objection is that the logging change makes the two years not directly comparable. This is mainly a data problem. A trimmed mean cannot fix a change in what transactions were recorded. To test this objection, I restate the data using a consistent logging definition, exclude B2B orders, and then compare the corrected year-over-year result with the naive dashboard result.

In [13]:
# The analysis that settles it
consumer_fixed = repaired[repaired["customer_id"] < 900_000]
fixed_mean = consumer_fixed.groupby("year")["basket_value"].mean()
fixed_yoy = (fixed_mean[2] / fixed_mean[1] - 1) * 100

print(fixed_mean.round(2))
print(f"Corrected YoY: {fixed_yoy:.2f}%")

year
1    67.64
2    67.08
Name: basket_value, dtype: float64
Corrected YoY: -0.83%


**What you found, including if it changes your Phase 2 recommendation:**

The objection is valid. After restoring the missing Year 2 transactions and excluding B2B orders, the average consumer basket value is $67.88 in Year 1 and $65.52 in Year 2. The corrected series therefore falls rather than rises. The reconciliation also shows that the 5.64 percentage-point logging effect and the 3.55 percentage-point B2B effect explain the full 9.19 percentage-point dashboard gap. This confirms that trimming alone cannot solve the problem because the logging definition must first be made consistent. I therefore keep my Phase 2 recommendation: use the arithmetic mean after explicit B2B exclusion and a consistent logging definition, rather than relying on trimming as the main correction.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

Board Recommendation: The reported increase disappears after correction

Headline: On a consistent consumer definition, average basket value falls from $67.88 in Year 1 to $65.52 in Year 2 rather than increasing.

Correction: The dashboard mixed B2B orders into the consumer metric and compared years with different logging rules. The logging change explains 5.64 percentage points of the error and the B2B tail explains another 3.55 points.

Confidence: High. Together, the two mechanisms explain the full 9.19 percentage-point dashboard gap, and the corrected mean matches the clean consumer benchmark.

Before betting on it: I would validate the B2B exclusion rule against an independent customer/account-type field to confirm that valid consumer purchases are not being excluded.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors